In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week9-assignment-1"). \
config("spark.sql.warehouse.dir", f"/user/itv027484/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [2]:
#Open a Spark Notebook and create a spark base dataframe by reading all thefiles under /public/sms/users folder. 
# Do check how many partitions are createdin your dataframe.

In [2]:
user_address_schema = StructType([
    StructField("city",StringType()),
    StructField("street",StringType()),
    StructField("state",StringType()),
    StructField("postal_code",StringType()),
])
user_schema = StructType ([
    StructField("user_id",LongType()),
    StructField("user_first_name",StringType()),
    StructField("user_last_name",StringType()),
    StructField("user_email",StringType()),
    StructField("user_gender",StringType()),
    StructField("user_phone_numbers",ArrayType(StringType())),
    StructField("user_address",user_address_schema)    
])

In [3]:
df1 = spark.read.format('json').schema(user_schema).load('/public/sms/users')

In [4]:
df1.show(5)

+-------+---------------+--------------+--------------------+-----------+--------------------+--------------------+
|user_id|user_first_name|user_last_name|          user_email|user_gender|  user_phone_numbers|        user_address|
+-------+---------------+--------------+--------------------+-----------+--------------------+--------------------+
| 200001|         Eirena|     Cutsforth|ecutsforth0@wisc.edu|     Female|[4197404036, 9173...|{Dallas, 8 Warrio...|
| 200002|          Marja|      Shopcott|mshopcott1@hexun.com|     Female|[9542037028, 2128...|{Joliet, 66 Prair...|
| 200003|           Dawn|       Tointon|  dtointon2@ucsd.edu|     Female|[9523035647, 2134...|{Shawnee Mission,...|
| 200004|          Goldi|        Leaman|     gleaman3@360.cn|     Female|[2027069459, 7042...|{Saint Paul, 7696...|
| 200005|       Brewster|      Hallagan|bhallagan4@livejo...|       Male|[8134746319, 2152...|{Albuquerque, 942...|
+-------+---------------+--------------+--------------------+-----------

In [6]:
#{"user_id":1,
# "user_first_name":"Lezley",
# "user_last_name":"D'Alessio",
# "user_email":"ldalessio0@google.com.au",
# "user_gender":"Male",
# "user_phone_numbers":["5639521582","8433335556","9193704732","8326122969"],
# "user_address":{"street":"28470 Di Loreto Point","city":"Albany","state":"New York","postal_code":"12222"}}

In [7]:
df1.printSchema()

root
 |-- user_id: long (nullable = true)
 |-- user_first_name: string (nullable = true)
 |-- user_last_name: string (nullable = true)
 |-- user_email: string (nullable = true)
 |-- user_gender: string (nullable = true)
 |-- user_phone_numbers: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- user_address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- street: string (nullable = true)
 |    |-- state: string (nullable = true)
 |    |-- postal_code: string (nullable = true)



In [8]:
df1.rdd.getNumPartitions()

3

In [9]:
## 10 files of 28 mb each . 4mb maxPartitionBytes . so 28+4 = 32 mb effectively. total 320mb. so for 128mb partitions, we need 3 files ( 320/128 = 2.5)

# DF API format

In [10]:
df1.count() # Total number of records in the dataframe

1000000

In [11]:
# how many users are from the state New York

#### How to filter on a column inside a nested column

In [12]:
df1.where(df1.user_address.state == "New York").count()

49576

In [13]:
df1.where(col("user_address.state") == "New York").count()

49576

In [14]:
df1.where(col("user_address")["state"] == "New York").count()

49576

In [15]:
df1.filter("user_address.state = 'New York'").count()

49576

In [16]:
# which state has maximum number of postal code

In [17]:
df1.select(df1.user_address.state.alias("state") , df1.user_address.postal_code.alias("postal_code")).groupBy("state").count().orderBy("count",ascending=False).show(5)

+----------+------+
|     state| count|
+----------+------+
|      null|108981|
|California| 97836|
|     Texas| 97236|
|   Florida| 73380|
|  New York| 49576|
+----------+------+
only showing top 5 rows



In [31]:
# which state has maximum number of distinct postal code

In [30]:
df1.select(df1.user_address.state.alias("state") , df1.user_address.postal_code.alias("postal_code")) \
.groupBy("state").agg(countDistinct("postal_code").alias("count")) \
.orderBy("count",ascending=False).show(5)

+----------+-----+
|     state|count|
+----------+-----+
|California|  206|
|     Texas|  205|
|   Florida|  155|
|  New York|  104|
|      Ohio|   68|
+----------+-----+
only showing top 5 rows



In [19]:
# which city has the most number of users

In [20]:
df1.select(df1.user_id , df1.user_address.city.alias("city")).groupBy("city").count().orderBy("count",ascending=False).show(5)

+-------------+------+
|         city| count|
+-------------+------+
|         null|108981|
|   Washington| 28504|
|      Houston| 18098|
|New York City| 15546|
|      El Paso| 14740|
+-------------+------+
only showing top 5 rows



In [21]:
## how many users have email domain as bizjournals.com

In [22]:
df1.filter(df1.user_email.like("%@bizjournals.com")).count()

2015

In [23]:
# how many users have 4 phone numbers mentioned

In [24]:
df1.filter(size("user_phone_numbers") == 4 ).count()

179041

In [25]:
df1.filter("size(user_phone_numbers) = 4" ).count()

179041

In [26]:
#how many users do not have any phone number mentioned

In [27]:
df1.filter("size(user_phone_numbers) = 0 or  size(user_phone_numbers) = -1  " ).count()

108981

In [32]:
#extracting columns from nested json and creating views

## SQL format

In [34]:
df1.withColumn("user_street",col("user_Address.street")) \
.withColumn("user_city",col("user_address.city")) \
.withColumn("user_state", col("user_Address.state")) \
.withColumn("user_postal_code", col("user_address.postal_code")) \
.withColumn("num_phn_numbers",size(col("user_phone_numbers"))).createOrReplaceTempView("users_vw")

In [37]:
spark.sql("select  * from users_vw limit 5")

user_id,user_first_name,user_last_name,user_email,user_gender,user_phone_numbers,user_address,user_street,user_city,user_state,user_postal_code,num_phn_numbers
200001,Eirena,Cutsforth,ecutsforth0@wisc.edu,Female,"[4197404036, 9173...","{Dallas, 8 Warrio...",8 Warrior Drive,Dallas,Texas,75358,4
200002,Marja,Shopcott,mshopcott1@hexun.com,Female,"[9542037028, 2128...","{Joliet, 66 Prair...",66 Prairieview Te...,Joliet,Illinois,60435,5
200003,Dawn,Tointon,dtointon2@ucsd.edu,Female,"[9523035647, 2134...","{Shawnee Mission,...",18 Ronald Regan Hill,Shawnee Mission,Kansas,66225,3
200004,Goldi,Leaman,gleaman3@360.cn,Female,"[2027069459, 7042...","{Saint Paul, 7696...",7696 Calypso Junc...,Saint Paul,Minnesota,55166,5
200005,Brewster,Hallagan,bhallagan4@livejo...,Male,"[8134746319, 2152...","{Albuquerque, 942...",942 Emmet Park,Albuquerque,New Mexico,87110,2


In [39]:
spark.sql("select  count(*) from users_vw")

count(1)
1000000


In [41]:
spark.sql("select count(distinct user_id ) from users_vw where user_state = 'New York' ")

count(DISTINCT user_id)
49576


In [42]:
spark.sql("select user_state, count(distinct user_postal_code ) as count from users_vw group by user_state order by count desc limit 1")

user_state,count
California,206


In [45]:
spark.sql("select user_city,count(distinct user_id) as count from users_vw group by user_city order by count desc limit 2")

user_city,count
null,108981
Washington,28504


In [46]:
spark.sql("select count(distinct user_id ) as count from users_vw  where user_email like '%@bizjournals.com'")

count
2015


In [48]:
spark.sql("select count(distinct user_id ) as count from users_vw  where num_phn_numbers in ( 0, -1) ")

count
108981


### Write the data from the base dataframe as it is to the disk, but write in parquetformat. Observe the number of files created, also the size of files.

In [5]:
df1.write.mode('overwrite').save('week9_json')

In [6]:
# [itv027484@g02 ~]$ hadoop fs -ls -h week9_json
# Found 4 items
# -rw-r--r--   3 itv027484 supergroup          0 2026-08-10 01:06 week9_json/_SUCCESS
# -rw-r--r--   3 itv027484 supergroup     25.9 M 2026-08-10 01:06 week9_json/part-00000-a4b186ba-b95a-46eb-be25-1a64bb962c67-c000.snappy.parquet
# -rw-r--r--   3 itv027484 supergroup     25.9 M 2026-08-10 01:06 week9_json/part-00001-a4b186ba-b95a-46eb-be25-1a64bb962c67-c000.snappy.parquet
# -rw-r--r--   3 itv027484 supergroup     13.1 M 2026-08-10 01:06 week9_json/part-00002-a4b186ba-b95a-46eb-be25-1a64bb962c67-c000.snappy.parquet
# [itv027484@g02 ~]$ 